In [1]:
import requests
from bs4 import BeautifulSoup

# Target site - a sandbox built specifically for scraping practice, legal and safe to scrape
url = "https://books.toscrape.com/"

# Fetch the raw HTML content of the page, same as we've done with API calls before
response = requests.get(url)

# Parse the raw HTML text into a navigable structure using BeautifulSoup
# 'html.parser' tells it which parsing engine to use (built into Python, no extra install needed)
soup = BeautifulSoup(response.text, 'html.parser')

# Print a readable, nicely-indented preview of the HTML structure
# Limited to first 2000 characters just to get a quick look, not flood the screen
print(soup.prettify()[:2000])

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!-->
<html class="no-js" lang="en-us">
 <!--<![endif]-->
 <head>
  <title>
   All products | Books to Scrape - Sandbox
  </title>
  <meta content="text/html; charset=utf-8" http-equiv="content-type"/>
  <meta content="24th Jun 2016 09:29" name="created"/>
  <meta content="" name="description"/>
  <meta content="width=device-width" name="viewport"/>
  <meta content="NOARCHIVE,NOCACHE" name="robots"/>
  <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
  <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
  <link href="static/oscar/favicon.ico" rel="shortcut icon"/>
  <link href="static/oscar/css/styles.css" rel="stylesheet" type="tex

In [5]:
# Find all book containers on the page - each book is wrapped in an <article> tag
# with the class "product_pod" (this is specific to this site's HTML structure,
# found by inspecting the page in a browser - not something you'd guess blindly)
books = soup.find_all('article', class_='product_pod')

# How many books did we find on this one page?
print(len(books))

# Look at just the first book's raw HTML, to understand its structure
print(books[0].prettify())

20
<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [3]:
# Look at just the product_price section of the first book to see price and availability
print(books[0].find('div', class_='product_price').prettify())

<div class="product_price">
 <p class="price_color">
  Â£51.77
 </p>
 <p class="instock availability">
  <i class="icon-ok">
  </i>
  In stock
 </p>
 <form>
  <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
   Add to basket
  </button>
 </form>
</div>



In [6]:
# Get the first book again for testing extraction logic
book = books[0]

# Title: it's stored in the 'title' attribute of the <a> tag inside <h3>, not the visible text
title = book.h3.a['title']

# Price: found inside the <p class="price_color"> tag, as text
price = book.find('p', class_='price_color').text

# Rating: stored as a CSS class on the <p> tag (e.g., "star-rating Three")
# We grab the second class name, which is the actual rating word (One, Two, Three, etc.)
rating = book.find('p', class_='star-rating')['class'][1]

# Availability: found inside <p class="instock availability">, as text (needs cleaning later)
availability = book.find('p', class_='instock availability').text.strip()

print(title)
print(price)
print(rating)
print(availability)

A Light in the Attic
Â£51.77
Three
In stock


In [7]:
# Loop through every book found on the page and extract the same 4 fields for each
book_data = []

for book in books:
    title = book.h3.a['title']
    price = book.find('p', class_='price_color').text
    rating = book.find('p', class_='star-rating')['class'][1]
    availability = book.find('p', class_='instock availability').text.strip()
    
    # Store each book's info as a dictionary, matching the pattern we used for API data
    book_data.append({
        'title': title,
        'price': price,
        'rating': rating,
        'availability': availability
    })

# Convert the list of dictionaries into a DataFrame, same pattern as Projects 3 & 4
import pandas as pd
df = pd.DataFrame(book_data)
df

,title,price,rating,availability
0,A Light in the Attic,Â£51.77,Three,In stock
1,Tipping the Velvet,Â£53.74,One,In stock
2,Soumission,Â£50.10,One,In stock
3,Sharp Objects,Â£47.82,Four,In stock
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock
5,The Requiem Red,Â£22.65,One,In stock
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,Four,In stock
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,Three,In stock
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,Four,In stock
9,The Black Maria,Â£52.15,One,In stock
